# Tesseract Baseline

Thin Colab/local entrypoint for OCR baseline execution.
Reusable logic stays in `src/`; this notebook only prepares runtime, uploads a sample, runs the pipeline, and exports artifacts.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

IS_COLAB = 'google.colab' in sys.modules
REPO_URL = os.environ.get('NFSE_REPO_URL', 'https://github.com/LuizCarlosGoulart/DolpII.git')
REPO_ROOT = Path(os.environ.get('NFSE_REPO_ROOT', '/content/nfse-extractor' if IS_COLAB else Path.cwd().resolve().parent))
PROJECT_ROOT = Path(os.environ.get('NFSE_PROJECT_ROOT', str(REPO_ROOT / 'nfse-extractor' if IS_COLAB else REPO_ROOT)))

CONFIG = {
    'clone_or_pull': os.environ.get('NFSE_CLONE_OR_PULL', '1') == '1',
    'bootstrap_runtime': os.environ.get('NFSE_BOOTSTRAP_RUNTIME', '1') == '1',
    'mount_drive': os.environ.get('NFSE_MOUNT_DRIVE', '0') == '1',
    'sample_path': os.environ.get('NFSE_SAMPLE_PATH', ''),
    'language': os.environ.get('NFSE_TESSERACT_LANGUAGE', 'por'),
    'raw_output_path': os.environ.get('NFSE_TESSERACT_RAW_OUTPUT', '/content/tesseract_raw_artifacts.json' if IS_COLAB else str(PROJECT_ROOT / 'artifacts' / 'tesseract_raw_artifacts.json')),
    'candidate_output_path': os.environ.get('NFSE_TESSERACT_CANDIDATE_OUTPUT', '/content/tesseract_field_candidates.json' if IS_COLAB else str(PROJECT_ROOT / 'artifacts' / 'tesseract_field_candidates.json')),
    'preview_limit': int(os.environ.get('NFSE_PREVIEW_LIMIT', '80')),
}

print(f'IS_COLAB: {IS_COLAB}')
print(f'REPO_ROOT: {REPO_ROOT}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')


In [ ]:
if IS_COLAB and CONFIG['clone_or_pull']:
    if REPO_ROOT.exists():
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull'], check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {PROJECT_ROOT}. Check NFSE_PROJECT_ROOT or the cloned repository layout.'
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if IS_COLAB and CONFIG['mount_drive']:
    from google.colab import drive
    drive.mount('/content/drive')

if IS_COLAB and CONFIG['bootstrap_runtime']:
    subprocess.run(['bash', str(PROJECT_ROOT / 'scripts' / 'colab_bootstrap.sh')], check=True)

print('Runtime ready')


In [ ]:
if IS_COLAB and not CONFIG['sample_path']:
    from google.colab import files

    uploaded = files.upload()
    if uploaded:
        first_file = next(iter(uploaded))
        uploaded_path = Path('/content') / first_file
        if not uploaded_path.exists():
            matches = sorted(Path.cwd().glob(first_file)) + sorted(Path("/content").rglob(first_file))
            uploaded_path = matches[-1] if matches else uploaded_path
        CONFIG['sample_path'] = str(uploaded_path)

if not CONFIG['sample_path']:
    raise ValueError('Set NFSE_SAMPLE_PATH or upload a sample PDF/image before running the pipeline.')

sample_path = Path(CONFIG['sample_path']).expanduser()
if not sample_path.exists():
    raise FileNotFoundError(f'Sample file not found: {sample_path}')

print(f'Sample: {sample_path}')


In [ ]:
from src.engines import TesseractExtractionAdapter
from src.export import write_extracted_elements_json
from src.ingestion import load_document
from src.preprocessing import preprocess_document

normalization_module = importlib.import_module("src.normalization")
ConfigDrivenOutputNormalizer = getattr(normalization_module, "ConfigDrivenOutputNormalizer", None)

document = load_document(sample_path)
preprocessed = preprocess_document(document)
adapter = TesseractExtractionAdapter(language=CONFIG['language'])
artifacts = adapter.extract_preprocessed(preprocessed)
raw_output_path = write_extracted_elements_json(artifacts, CONFIG['raw_output_path'])

candidates = []
candidate_output_path = None
if ConfigDrivenOutputNormalizer is not None:
    normalizer = ConfigDrivenOutputNormalizer()
    candidates = normalizer.normalize(document, artifacts)
    candidate_payload = [candidate.model_dump(mode="json") for candidate in candidates]
    candidate_output_path = Path(CONFIG['candidate_output_path'])
    candidate_output_path.parent.mkdir(parents=True, exist_ok=True)
    candidate_output_path.write_text(json.dumps(candidate_payload, indent=2, ensure_ascii=False), encoding="utf-8")

print(f'Document: {document.document_id}')
print(f'Media type: {document.media_type}')
print(f'Pages: {preprocessed.metadata["page_count"]}')
print(f'Raw artifacts: {len(artifacts)}')
print(f'Field candidates: {len(candidates)}')
print(f'Raw output: {raw_output_path}')
print(f'Candidate output: {candidate_output_path}')


In [ ]:
print('Raw OCR preview')
for item in artifacts[: CONFIG['preview_limit']]:
    print(
        f'{item.text!r} | conf={item.confidence} | page={item.page_number} | bbox={item.bounding_box}'
    )

if candidates:
    print('\nField candidate preview')
    for candidate in candidates[: CONFIG['preview_limit']]:
        print(
            f'{candidate.field_name} = {candidate.value!r} | '
            f'conf={candidate.confidence} | '
            f'section={candidate.metadata.get("section_name")} | '
            f'label={candidate.metadata.get("label_text")}'
        )
